In [1]:

# Libraries
import pandas as pd
import numpy as np
import plotly.express as px

# Load cleaned dataset
df = pd.read_csv(
    "../data/processed/sales_clean.csv",
    sep=";",
    decimal=",",
    parse_dates=["order_date"]
)

# Basic configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Create a dataset with completed orders only
sales_df = df[df["status"] == "Completed"].copy()

print("Dataset loaded successfully!")
print(f"Total records: {len(df)}")
print(f"Completed orders: {len(sales_df)}")
print(f"Cancelled orders: {len(df[df['status'] == 'Cancelled'])}")


Dataset loaded successfully!
Total records: 1000
Completed orders: 875
Cancelled orders: 125


In [3]:

# Dataset overview
print("Dataset dimensions:", df.shape)

print("\nColumn information:")
display(df.info())

print("\nStatistical summary:")
display(df.describe(include="all"))

print("\nUnique values:")
for column in ["category", "city", "status", "product_name"]:
    print(f"{column}: {df[column].nunique()} unique values")


Dataset dimensions: (1000, 12)

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      1000 non-null   str           
 1   order_date    1000 non-null   datetime64[us]
 2   customer_id   1000 non-null   str           
 3   product_id    1000 non-null   str           
 4   product_name  1000 non-null   str           
 5   category      1000 non-null   str           
 6   city          1000 non-null   str           
 7   quantity      1000 non-null   int64         
 8   unit_price    1000 non-null   float64       
 9   discount      1000 non-null   float64       
 10  status        1000 non-null   str           
 11  revenue       1000 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(7)
memory usage: 93.9 KB


None


Statistical summary:


,order_id,order_date,customer_id,product_id,product_name,category,city,quantity,unit_price,discount,status,revenue
count,1000,1000,1000,1000,1000,1000,1000,"1,000.00","1,000.00","1,000.00",1000,"1,000.00"
unique,1000,NaN,286,11,11,5,6,NaN,NaN,NaN,2,NaN
top,ORD0001,NaN,CUST0216,P007,Jeans,Electronics,Porto Alegre,NaN,NaN,NaN,Completed,NaN
freq,1,NaN,9,105,105,263,208,NaN,NaN,NaN,875,NaN
mean,NaN,2024-06-30 00:25:55.200000,NaN,NaN,NaN,NaN,NaN,3.05,853.15,0.06,NaN,"2,491.64"
min,NaN,2024-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,1.00,58.54,0.00,NaN,50.41
25%,NaN,2024-04-03 00:00:00,NaN,NaN,NaN,NaN,NaN,2.00,158.21,0.00,NaN,355.32
50%,NaN,2024-06-30 00:00:00,NaN,NaN,NaN,NaN,NaN,3.00,350.76,0.05,NaN,"1,079.62"
75%,NaN,2024-09-29 00:00:00,NaN,NaN,NaN,NaN,NaN,4.00,"1,110.65",0.10,NaN,"3,083.31"
max,NaN,2024-12-31 00:00:00,NaN,NaN,NaN,NaN,NaN,5.00,"3,847.69",0.20,NaN,"18,721.25"



Unique values:
category: 5 unique values
city: 6 unique values
status: 2 unique values
product_name: 11 unique values


In [2]:

# Revenue by category
revenue_by_category = (
    sales_df.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(revenue_by_category)

# Visualization
fig = px.bar(
    revenue_by_category,
    x="category",
    y="revenue",
    title="Total Revenue by Product Category",
    labels={
        "category": "Product Category",
        "revenue": "Revenue (R$)"
    },
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Product Category",
    yaxis_title="Revenue (R$)",
    template="plotly_white"
)

fig.show()


,category,revenue
0,Electronics,"1,373,727.98"
1,Furniture,"480,158.42"
2,Home & Kitchen,"176,261.83"
3,Sports,"88,731.31"
4,Clothing,"52,154.97"


In [3]:

# Monthly revenue
monthly_revenue = (
    sales_df.set_index("order_date")
    .resample("MS")["revenue"]
    .sum()
    .reset_index()
)

monthly_revenue["month"] = (
    monthly_revenue["order_date"].dt.strftime("%b")
)

display(monthly_revenue)

# Visualization
fig = px.line(
    monthly_revenue,
    x="order_date",
    y="revenue",
    markers=True,
    title="Monthly Revenue Trend - 2024",
    labels={
        "order_date": "Month",
        "revenue": "Revenue (R$)"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Month",
    yaxis_title="Revenue (R$)"
)

fig.show()


,order_date,revenue,month
0,2024-01-01,"156,926.18",Jan
1,2024-02-01,"143,904.63",Feb
2,2024-03-01,"216,676.65",Mar
3,2024-04-01,"242,404.40",Apr
4,2024-05-01,"151,576.29",May
5,2024-06-01,"174,184.78",Jun
6,2024-07-01,"175,396.34",Jul
7,2024-08-01,"187,139.44",Aug
8,2024-09-01,"193,755.14",Sep
9,2024-10-01,"240,913.84",Oct


In [4]:

# Revenue by city
revenue_by_city = (
    sales_df.groupby("city")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(revenue_by_city)

# Visualization
fig = px.bar(
    revenue_by_city,
    x="city",
    y="revenue",
    color="revenue",
    title="Revenue by City",
    labels={
        "city": "City",
        "revenue": "Revenue (R$)"
    },
    text_auto=".2s"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="City",
    yaxis_title="Revenue (R$)",
    showlegend=False
)

fig.show()


,city,revenue
0,Porto Alegre,"471,755.54"
1,Curitiba,"465,471.98"
2,Sao Paulo,"421,835.34"
3,Belo Horizonte,"408,744.44"
4,Rio de Janeiro,"381,404.82"
5,Unknown,"21,822.39"


In [5]:

# Top 5 products by quantity sold
top_products = (
    sales_df.groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

display(top_products)

# Visualization
fig = px.bar(
    top_products.sort_values("quantity"),
    x="quantity",
    y="product_name",
    orientation="h",
    title="Top 5 Best-Selling Products",
    labels={
        "product_name": "Product",
        "quantity": "Units Sold"
    },
    text="quantity"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Units Sold",
    yaxis_title="Product"
)

fig.show()


,product_name,quantity
0,Office Chair,283
1,Jeans,273
2,Laptop,269
3,Coffee Maker,257
4,Yoga Mat,254


In [6]:

# Order status distribution
status_distribution = (
    df["status"]
    .value_counts()
    .reset_index()
)

status_distribution.columns = ["status", "orders"]

display(status_distribution)

# Cancellation rate
total_orders = len(df)
cancelled_orders = len(df[df["status"] == "Cancelled"])

cancellation_rate = (
    cancelled_orders / total_orders * 100
)

print(f"Total Orders: {total_orders}")
print(f"Cancelled Orders: {cancelled_orders}")
print(f"Cancellation Rate: {cancellation_rate:.2f}%")

# Visualization
fig = px.pie(
    status_distribution,
    names="status",
    values="orders",
    title="Order Status Distribution",
    hole=0.4
)

fig.update_layout(template="plotly_white")

fig.show()


,status,orders
0,Completed,875
1,Cancelled,125


Total Orders: 1000
Cancelled Orders: 125
Cancellation Rate: 12.50%


In [7]:

# Key business findings

best_category = revenue_by_category.loc[
    revenue_by_category["revenue"].idxmax()
]

best_city = revenue_by_city.loc[
    revenue_by_city["revenue"].idxmax()
]

best_month = monthly_revenue.loc[
    monthly_revenue["revenue"].idxmax()
]

best_product = top_products.loc[
    top_products["quantity"].idxmax()
]

print("=" * 45)
print("SALES INTELLIGENCE - KEY FINDINGS")
print("=" * 45)

print(f"\nTotal Revenue: R$ {sales_df['revenue'].sum():,.2f}")

print(
    f"Top Revenue Category: "
    f"{best_category['category']}"
)

print(
    f"Top Revenue City: "
    f"{best_city['city']}"
)

print(
    f"Best Revenue Month: "
    f"{best_month['month']}"
)

print(
    f"Best-Selling Product: "
    f"{best_product['product_name']}"
)

print(
    f"Cancellation Rate: "
    f"{cancellation_rate:.2f}%"
)

print("=" * 45)


SALES INTELLIGENCE - KEY FINDINGS

Total Revenue: R$ 2,171,034.51
Top Revenue Category: Electronics
Top Revenue City: Porto Alegre
Best Revenue Month: Apr
Best-Selling Product: Office Chair
Cancellation Rate: 12.50%


In [8]:

# Import metrics functions
import sys
sys.path.insert(0, "../src")

from metrics import (
    load_data,
    calculate_total_revenue,
    calculate_total_orders,
    calculate_average_order_value,
    calculate_unique_customers,
    calculate_cancellation_rate
)

# Load data
metrics_df = load_data()

# Calculate metrics
revenue = calculate_total_revenue(metrics_df)
orders = calculate_total_orders(metrics_df)
aov = calculate_average_order_value(metrics_df)
customers = calculate_unique_customers(metrics_df)
cancellation = calculate_cancellation_rate(metrics_df)

# Display results
print(f"Total Revenue: R$ {revenue:,.2f}")
print(f"Total Orders: {orders}")
print(f"Average Order Value: R$ {aov:,.2f}")
print(f"Unique Customers: {customers}")
print(f"Cancellation Rate: {cancellation:.2f}%")

# Basic validation
assert orders == 875
assert cancellation == 12.50
assert aov == round(revenue / orders, 2)

print("\nAll metric validations passed!")


Total Revenue: R$ 2,171,034.51
Total Orders: 875
Average Order Value: R$ 2,481.18
Unique Customers: 280
Cancellation Rate: 12.50%

All metric validations passed!
